In [8]:
from search_utils import AverageCellPerturbationSearch, search_to_df
import pickle
from tqdm import tqdm
import pandas as pd
import numpy as np

In [24]:
def run_beta_tuning_experiment(search_env):
    """
    Runs a refined experiment to find the optimal beta value by testing
    across multiple conversion pairs.
    """
    # --- 1. Experiment Setup ---
    beta_values_to_test = [10.0, 20.0, 30.0]
    num_runs_per_condition = 5  # Number of stochastic runs for each (beta, pair) combo

    # --- IMPORTANT: Define your conversion pairs here ---
    # Choose 2 or 3 representative problems for robust tuning.
    conversion_pairs_to_test = [
        ('CVCL_1097', 'CVCL_0292'), # e.g., A known short-distance conversion
        ('CVCL_C466', 'CVCL_0292'), # e.g., A known long-distance conversion
        ('CVCL_1097', 'CVCL_1716')  # e.g., A conversion of biological interest
    ]
    # ----------------------------------------------------

    print(f"Starting beta tuning experiment.")
    print(f"Beta values: {beta_values_to_test}")
    print(f"Testing on {len(conversion_pairs_to_test)} conversion pairs.")
    
    results_log = []

    # --- 2. Main Experiment Loop ---
    # Use tqdm for a nice progress bar
    for beta in tqdm(beta_values_to_test, desc="Testing Beta Values"):
        
        # Store scores for this beta across all conversion pairs
        all_scores_for_this_beta = []

        for start_cl, end_cl in conversion_pairs_to_test:
            
            run_scores = []
            for i in range(num_runs_per_condition):
                
                # Run your probabilistic search
                search_results = search_env.probabilistic_search(
                    search_id=f'beta_tune_{beta}_{start_cl}_to_{end_cl}_run_{i}',
                    starting_cl=start_cl,
                    ending_cl=end_cl,
                    beta=beta,
                    n_paths=100,
                    n_steps=5
                )
                
                results_df = search_to_df(search_results)
                
                # --- MODIFIED: Your Evaluation Metric Here ---
                # Using the average progress percentage of the top 5 paths. Higher is better.
                if not results_df.empty and 'final_distance' in results_df.columns and 'starting_distance' in results_df.columns:
                    # Calculate progress percentage for each path
                    # Add a small epsilon to avoid division by zero if starting_distance is 0
                    results_df['progress_perc'] = (results_df['starting_distance'] - results_df['final_distance']) / (results_df['starting_distance'] + 1e-9)
                    
                    # Sort by progress and take the top 5
                    top_5_paths = results_df.nlargest(5, 'progress_perc')
                    
                    # The score is the average progress of these top paths
                    if not top_5_paths.empty:
                        score = top_5_paths['progress_perc'].mean()
                    else:
                        score = 0.0 # No paths found, so progress is 0
                else:
                    score = 0.0 # No paths found, so progress is 0
                # ------------------------------------
                    
                run_scores.append(score)
            
            # Average the scores for this specific conversion pair
            avg_pair_score = np.mean(run_scores)
            all_scores_for_this_beta.append(avg_pair_score)

        # Calculate the overall average score for this beta across all pairs
        overall_avg_score = np.mean(all_scores_for_this_beta)
        overall_std_dev = np.std(all_scores_for_this_beta)
        
        results_log.append({
            'beta': beta,
            'overall_avg_score': overall_avg_score,
            'overall_std_dev': overall_std_dev
        })

    # --- 3. Analyze and Print Final Results ---
    final_results_df = pd.DataFrame(results_log).set_index('beta')
    print("\n--- Final Tuning Summary ---")
    print(final_results_df)

    # --- MODIFIED: Find and announce the best beta (assuming higher score is better) ---
    best_beta = final_results_df['overall_avg_score'].idxmax()
    print(f"\n🏆 Optimal beta found: {best_beta}")
    
    return final_results_df

In [21]:
with open('../data_and_models/cleaned_data.pkl', 'rb') as f:
    df = pickle.load(f)

In [22]:
search_env = AverageCellPerturbationSearch(df)

In [23]:
run_beta_tuning_experiment(search_env)

Starting beta tuning experiment.
Beta values: [1.0, 2.5, 5.0, 7.5, 10.0]
Testing on 3 conversion pairs.


Testing Beta Values: 100%|██████████| 5/5 [02:03<00:00, 24.79s/it]


--- Final Tuning Summary ---
      overall_avg_score  overall_std_dev
beta                                    
1.0            0.467723         0.014427
2.5            0.518907         0.013706
5.0            0.548405         0.024054
7.5            0.558588         0.024201
10.0           0.559121         0.020825

🏆 Optimal beta found: 10.0


,overall_avg_score,overall_std_dev
beta,,
1.0,0.467723,0.014427
2.5,0.518907,0.013706
5.0,0.548405,0.024054
7.5,0.558588,0.024201
10.0,0.559121,0.020825


In [25]:
run_beta_tuning_experiment(search_env)

Starting beta tuning experiment.
Beta values: [10.0, 20.0, 30.0]
Testing on 3 conversion pairs.


Testing Beta Values: 100%|██████████| 3/3 [01:14<00:00, 24.79s/it]


--- Final Tuning Summary ---
      overall_avg_score  overall_std_dev
beta                                    
10.0           0.556159         0.021144
20.0           0.542477         0.010043
30.0           0.532478         0.007094

🏆 Optimal beta found: 10.0


,overall_avg_score,overall_std_dev
beta,,
10.0,0.556159,0.021144
20.0,0.542477,0.010043
30.0,0.532478,0.007094
